In [ ]:
# 第9〜11章 統合演習：Google Colabの1セルで実行
# 元の住宅データはscikit-learnから自動取得（CSVアップロード不要）
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style='whitegrid')

# 9章：データの取得・理解・前処理
raw = fetch_california_housing(as_frame=True)
rename = {'MedInc':'所得', 'HouseAge':'築年数', 'AveRooms':'部屋数',
          'AveBedrms':'寝室数', 'Population':'地域人口', 'AveOccup':'世帯人数',
          'Latitude':'緯度', 'Longitude':'経度'}
df = raw.data.rename(columns=rename).copy()
df['住宅価格'] = raw.target.to_numpy()
print('【9章：前処理】')
print('取得したデータ:', df.shape, '欠損数:', int(df.isna().sum().sum()))
print('築年数52の件数:', int((df['築年数'] == 52).sum()))
print('価格5.00001の件数:', int(np.isclose(df['住宅価格'], 5.00001, rtol=0, atol=1e-8).sum()))

# 教材と同じ順番・条件で除外（価格は浮動小数点誤差を考慮）
n_before = len(df)
age_mask = df['築年数'] == 52
print('築年数52で除外:', int(age_mask.sum()))
df = df.loc[~age_mask].copy()
price_mask = np.isclose(df['住宅価格'], 5.00001, rtol=0, atol=1e-8)
print('住宅価格上限値で除外:', int(price_mask.sum()))
df = df.loc[~price_mask].copy()

# 教材の中間計算は部屋数/世帯人数と同値になる（世帯人数0は欠損として扱う）
occup = df['世帯人数'].replace(0, np.nan)
df['部屋数/人'] = df['部屋数'] / occup
df['寝室数/人'] = df['寝室数'] / occup
features7 = ['所得','築年数','地域人口','緯度','経度','部屋数/人','寝室数/人']
df = df[features7 + ['住宅価格']].replace([np.inf, -np.inf], np.nan).dropna()
print(f'前処理後: {df.shape}（削除: {n_before-len(df)}行）')
if df.shape != (18572, 8):
    print('注意: 教材の18,572行とは異なります。データ・除外条件を確認してください。分析は続行します。')
print(df.head().to_string(index=False))
print('\n基本統計量:')
print(df.describe().round(3).to_string())
df.to_csv('california_housing_cleansing.csv', index=False, encoding='utf-8-sig')

# 10〜11章：すべて同じ行・同じ学習/テスト分割で比較
X = df[features7]
y = df['住宅価格']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)
models = {
    '10章: 標準化なし (7変数)': (LinearRegression(), features7),
    '11章: 標準化あり (7変数)': (make_pipeline(StandardScaler(), LinearRegression()), features7),
    '11章: 変数削減 (5変数)': (make_pipeline(StandardScaler(), LinearRegression()),
                         ['所得','築年数','地域人口','緯度','部屋数/人'])
}
results = []
fitted = {}
print('\n【10〜11章：モデル学習・評価】')
for name, (model, columns) in models.items():
    model.fit(X_train[columns], y_train)
    predicted = model.predict(X_test[columns])
    results.append({'モデル':name, '学習R2':model.score(X_train[columns], y_train),
                    'テストR2':r2_score(y_test, predicted),
                    'MAE':mean_absolute_error(y_test, predicted),
                    'RMSE':np.sqrt(mean_squared_error(y_test, predicted))})
    fitted[name] = (model, columns)
result_df = pd.DataFrame(results)
print(result_df.round(4).to_string(index=False))
result_df.to_csv('housing_model_comparison.csv', index=False, encoding='utf-8-sig')

# 標準化後の係数：スケールはそろうが因果効果とは解釈しない
scaled_model = fitted['11章: 標準化あり (7変数)'][0].named_steps['linearregression']
coef_df = pd.DataFrame({'特徴量': features7, '標準化後の係数': scaled_model.coef_})
print('\n【標準化モデルの回帰係数】')
print(coef_df.round(4).to_string(index=False))
print('\n【説明変数の相関係数】')
print(X.corr().round(2).to_string())

# 新規データの予測（7変数モデル）
X_new = pd.DataFrame([[8,41,500,37,-120,1,0.2],
                      [2,10,2000,38,-122,1.5,0.5],
                      [1,25,1000,38,-121,2,1]], columns=features7)
print('\n【住宅価格の予測】単位: 10万ドル')
for name in ['10章: 標準化なし (7変数)', '11章: 標準化あり (7変数)']:
    model, cols = fitted[name]
    print(name, np.round(model.predict(X_new[cols]), 4))

# 可視化（英語ラベルで日本語フォント依存を回避）
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].bar(['No scale 7','Scaled 7','Scaled 5'], result_df['テストR2'])
axes[0].set(title='Test R-squared', ylabel='R-squared')
axes[1].barh(['Income','House age','Population','Latitude','Longitude','Rooms/person','Beds/person'], scaled_model.coef_)
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set(title='Coefficients after standardization', xlabel='Coefficient')
fig.tight_layout()
plt.savefig('housing_regression_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('\n保存完了: california_housing_cleansing.csv, housing_model_comparison.csv, housing_regression_results.png')
